# Phase 3: prompting strategies on HallusionBench

Runs **chain-of-thought (CoT)**, **Chain-of-Verification (CoVe)** and **self-consistency (SC)** on a fixed sample of whole sets,
then compares them with the zero-shot results from Phase 2 under identical scoring.

**Use one notebook per model** (Qwen2.5-VL first, then InternVL3). Run cells in order. Cells 6a, 6b and 6c are slow and resume-safe:
if Colab disconnects, re-run Cells 1 to 5, then the same cell again, and it carries on where it stopped.

Upload your Phase 2 results file for this model (for example `qwen2.5vl_results.json`) to the Colab file panel before Cell 7.

## 1. Setup and settings

In [1]:
# Run once. Do NOT add -U pillow (it broke the InternVL notebook before). Restart the session if pip asks you to.
!pip -q install "bitsandbytes>=0.46.1" datasets accelerate

In [2]:
import json, re, time, random, collections, statistics, gc
import torch

# ---------- settings you may change ----------
MODEL_TAG   = "qwen25vl"                  # prefix for output files: "qwen25vl" or "internvl3"
ZS_FILE     = "qwen2.5vl_results.json"    # this model's Phase 2 results file (uploaded to Colab)
SEED        = 0                           # keep identical in BOTH notebooks so both models get the same sample
FRACTION    = 0.25                        # fraction of sets drawn per subcategory
MIN_SETS    = 2                           # at least this many sets per subcategory
K_SC        = 5                           # samples for self-consistency
SC_TEMP     = 0.7                         # sampling temperature for self-consistency
N_VERIF     = 3                           # verification questions for CoVe
COT_TOKENS  = 200                       # max new tokens for a reasoning answer
# ---------------------------------------------

## 2. Load the model

The default below is Qwen2.5-VL-7B in 4-bit. **If you already have a working loader and generate function from your Phase 2 notebook, use that instead.**
All the rest of this notebook needs is a function

`gen(image, text, max_new_tokens=16, temperature=0.0) -> str`

where `image` is a PIL image or `None` (text-only question), and `temperature=0.0` means greedy decoding.
For InternVL3, use the second block (it is commented out) and comment out the Qwen block.

In [ ]:
# ----- Qwen2.5-VL-7B (4-bit) -----
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=768*28*28)  # same limits as Phase 2

def gen(image, text, max_new_tokens=16, temperature=0.0):
    content = ([{"type": "image"}] if image is not None else []) + [{"type": "text", "text": text}]
    prompt = processor.apply_chat_template([{"role": "user", "content": content}], tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt], images=[image] if image is not None else None, return_tensors="pt").to(model.device)
    kw = dict(max_new_tokens=max_new_tokens)
    kw.update(dict(do_sample=True, temperature=temperature, top_p=0.9) if temperature > 0 else dict(do_sample=False))
    with torch.no_grad():
        out = model.generate(**inputs, **kw)
    return processor.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()

def gen_many(image, text, n, max_new_tokens=200, temperature=0.7):
    content = ([{"type": "image"}] if image is not None else []) + [{"type": "text", "text": text}]
    prompt = processor.apply_chat_template([{"role": "user", "content": content}], tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt], images=[image] if image is not None else None, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature,
                             top_p=0.9, num_return_sequences=n)
    return [t.strip() for t in processor.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [3]:
# ----- InternVL3-8B-hf (4-bit): uncomment this block and comment out the Qwen block above -----
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
MODEL_ID = "OpenGVLab/InternVL3-8B-hf"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID)

def gen(image, text, max_new_tokens=16, temperature=0.0):
    content = ([{"type": "image"}] if image is not None else []) + [{"type": "text", "text": text}]
    prompt = processor.apply_chat_template([{"role": "user", "content": content}], tokenize=False, add_generation_prompt=True)
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device, torch.float16)
    kw = dict(max_new_tokens=max_new_tokens)
    kw.update(dict(do_sample=True, temperature=temperature, top_p=0.9) if temperature > 0 else dict(do_sample=False))
    with torch.no_grad():
        out = model.generate(**inputs, **kw)
    return processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()


def gen_many(image, text, n, max_new_tokens=200, temperature=0.7):
    content = ([{"type": "image"}] if image is not None else []) + [{"type": "text", "text": text}]
    prompt = processor.apply_chat_template([{"role": "user", "content": content}], tokenize=False, add_generation_prompt=True)
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device, torch.float16)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature,
                             top_p=0.9, num_return_sequences=n)
    return [t.strip() for t in processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)]

def gen_many_chunked(image, text, n, chunk=2, max_new_tokens=200, temperature=0.7):
    out = []
    while len(out) < n:
        out += gen_many(image, text, min(chunk, n - len(out)), max_new_tokens, temperature)
        torch.cuda.empty_cache()
    return out
# If this block errors (InternVL loading was fiddly in Phase 2), use your working Phase 2 loader and generate function instead.

Loading weights:   0%|          | 0/781 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [4]:
# Quick test: one short answer
print(gen(None, "Is the sky blue? Please answer with only Yes or No."))

Yes.


## 3. Load the dataset and draw the sample

Whole sets are sampled (all questions that share a category, subcategory and `set_id`), stratified by subcategory, with a fixed seed.
This keeps every question pair and every figure intact so qAcc and fAcc stay computable. The sample is identical for every model and strategy.

In [5]:
from datasets import load_dataset
raw = load_dataset("lmms-lab/HallusionBench")
dsets = {"image": raw["image"], "non_image": raw["non_image"]}
S = lambda x: str(x).strip()

def meta(r):
    return dict(category=S(r.get("category", "")), subcategory=S(r.get("subcategory", "")), set_id=S(r.get("set_id", "")),
                figure_id=S(r.get("figure_id", "")), question_id=S(r.get("question_id", "")))
rowkey = lambda sp, m: "|".join([sp, m["category"], m["subcategory"], m["set_id"], m["figure_id"], m["question_id"]])
setkey = lambda m: (m["category"], m["subcategory"], m["set_id"])

# index of every question (no image decoding here)
index = []
for sp, d in dsets.items():
    light = d.remove_columns([c for c in ["image"] if c in d.column_names])
    for i, r in enumerate(light):
        index.append((sp, i, meta(r)))
keys = [rowkey(sp, m) for sp, i, m in index]
print("questions:", len(index), "| unique keys:", len(set(keys)))
if len(set(keys)) != len(index):
    print("WARNING: some row keys are not unique; tell Claude before continuing.")

by_sub = collections.defaultdict(set)
for sp, i, m in index:
    by_sub[(m["category"], m["subcategory"])].add(setkey(m))
rng = random.Random(SEED)
chosen = set()
for sub, sets in sorted(by_sub.items()):
    sets = sorted(sets)
    k = min(len(sets), max(MIN_SETS, round(FRACTION * len(sets))))
    chosen |= set(rng.sample(sets, k))
items = [(sp, i, m) for sp, i, m in index if setkey(m) in chosen]

cnt = collections.Counter((m["category"], m["subcategory"]) for sp, i, m in items)
print("sampled sets:", len(chosen), "| sampled questions:", len(items), "of", len(index))
for sub, n in sorted(cnt.items()):
    print(f"  {sub[0]}_{sub[1]:<8} {n:4d} questions, {sum(1 for s in chosen if s[:2]==sub)} sets")
sample_keys = {rowkey(sp, m) for sp, i, m in items}

def get_item(sp, i):
    r = dsets[sp][i]
    img = r.get("image")
    return r, (img.convert("RGB") if img is not None else None)

questions: 1129 | unique keys: 1129
sampled sets: 42 | sampled questions: 285 of 1129
  VD_figure     22 questions, 5 sets
  VD_illusion   40 questions, 8 sets
  VD_math       24 questions, 4 sets
  VD_ocr        20 questions, 5 sets
  VD_video      52 questions, 5 sets
  VS_chart      43 questions, 6 sets
  VS_map        24 questions, 3 sets
  VS_ocr        18 questions, 3 sets
  VS_table      42 questions, 3 sets


## 4. Prompts, strategies and answer parsing

Every strategy returns the same fields: `raw_response`, `model_prediction` (`"1"` = Yes, `"0"` = No, `"-1"` = could not parse, scored wrong,
as in Phase 2) plus a few extras for the error analysis.

In [6]:
to01 = lambda a: {"yes": "1", "no": "0"}.get(a, "-1")

def parse_short(t):
    """Short answers: must start with yes/no (same rule as Phase 2)."""
    t = t.strip().lower().lstrip("*\"' ")
    return "yes" if t.startswith("yes") else "no" if t.startswith("no") else "unclear"

def parse_final(t):
    """Reasoning answers: take the last 'Final answer: Yes/No'. Fallback: last standalone yes/no in the last 60 characters."""
    m = re.findall(r"final answer\s*[:\-]?\s*[\*\"']*\s*(yes|no)\b", t, flags=re.I)
    if m:
        return m[-1].lower(), "format"
    tail = re.findall(r"\b(yes|no)\b", t.lower()[-60:])
    return (tail[-1], "fallback") if tail else ("unclear", "none")

FORMAT = "Finish with a last line in exactly this form: 'Final answer: Yes' or 'Final answer: No'."
def cot_prompt(q, img):
    if img is not None:
        return (f"{q}\nLook at the image carefully. In at most 3 short sentences, describe the details relevant to the question "
                f"and reason to an answer. {FORMAT}")
    return f"{q}\nReason in at most 3 short sentences. {FORMAT}"

# ---- zero-shot style short answer (used inside CoVe) ----
def short(q, img):
    return gen(img, f"{q} Please answer with only Yes or No.", 8)

# ---- 1) Chain-of-thought ----
def strat_cot(q, img):
    raw = gen(img, cot_prompt(q, img), COT_TOKENS)
    a, mode = parse_final(raw)
    return dict(raw_response=raw, model_prediction=to01(a), parse_mode=mode)

# ---- 2) Chain-of-Verification ----
def strat_cove(q, img):
    draft = parse_short(short(q, img))
    plan = gen(img, f"Question: {q}\nA draft answer is: {draft}.\n"
                    f"Write {N_VERIF} short factual questions about what can be seen in the image (or about the facts needed) "
                    f"that would check whether this draft answer is correct. One question per line, no numbering, no answers.", 120)
    vqs = [l.strip(" -*0123456789.)") for l in plan.splitlines() if "?" in l][:N_VERIF]
    vas = [gen(img, vq + " Answer briefly.", 60) for vq in vqs]          # each check answered independently of the draft
    if vqs:
        qa = "\n".join(f"Q: {a}\nA: {b}" for a, b in zip(vqs, vas))
        final = gen(img, f"Question: {q}\nVerification questions and answers:\n{qa}\n"
                         f"Using the image and these verification answers (ignore the earlier draft if it conflicts), "
                         f"answer the original question with only Yes or No.", 8)
        a = parse_short(final)
    else:
        final, a = "", draft
    return dict(raw_response=final, model_prediction=to01(a), draft=draft, verification=list(zip(vqs, vas)))

# ---- 3) Self-consistency (K sampled CoT answers, majority vote) ----
_warned = False
def strat_sc(q, img):
    global _warned
    p = cot_prompt(q, img)
    try:
        samples = gen_many_chunked(img, p, K_SC, 2, COT_TOKENS, SC_TEMP)
    except Exception as e:
        if not _warned:
            print("batched sampling failed, using slow fallback:", type(e).__name__, str(e)[:200]); _warned = True
        samples = [gen(img, p, COT_TOKENS, temperature=SC_TEMP) for _ in range(K_SC)]
    votes = [parse_final(s)[0] for s in samples]
    c = collections.Counter(v for v in votes if v != "unclear")
    if c:
        top = c.most_common()
        tied = [l for l, n in top if n == top[0][1]]
        a = next(v for v in votes if v in tied)
    else:
        a = "unclear"
    return dict(raw_response=samples[0], model_prediction=to01(a), votes=votes, samples=samples)

## 5. Runner (resume-safe)

`run(strategy, name)` writes `{MODEL_TAG}_{name}.json` every 10 questions and skips questions already done.
`run(strategy, name, limit=10)` is a **timing test**: it runs 10 questions, prints seconds per question and saves nothing.

In [7]:
def run(strategy, name, limit=None):
    path = f"{MODEL_TAG}_{name}.json"
    done = {}
    if not limit:
        try:
            done = {r["key"]: r for r in json.load(open(path))}
        except FileNotFoundError:
            pass
    todo = [x for x in items if rowkey(x[0], x[2]) not in done]
    if limit:
        todo = todo[:limit]
    t0 = time.time()
    for n, (sp, i, m) in enumerate(todo, 1):
        r, img = get_item(sp, i)
        try:
            out = strategy(S(r["question"]), img)
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            out = dict(raw_response="OOM", model_prediction="-1")
        row = dict(key=rowkey(sp, m), split=sp, **m, question=S(r["question"]), gt_answer=S(r["gt_answer"]), **out)
        done[row["key"]] = row
        if not limit and n % 10 == 0:
            json.dump(list(done.values()), open(path, "w"))
            print(f"{len(done)}/{len(items)}  {(time.time()-t0)/n:.1f}s per question", end="\r")
    if limit:
        sec = (time.time() - t0) / max(1, len(todo))
        print(f"{sec:.1f}s per question -> full sample of {len(items)} would take about {sec*len(items)/60:.0f} minutes")
    else:
        json.dump(list(done.values()), open(path, "w"))
        print(f"\nsaved {len(done)} rows to {path}")

In [8]:
# Timing tests (10 questions each, nothing saved). Use these to decide whether to shrink FRACTION or K_SC before the long runs.
run(strat_cot, "cot", limit=10)
run(strat_cove, "cove", limit=10)
run(strat_sc, "sc", limit=5)

13.8s per question -> full sample of 285 would take about 66 minutes
60.2s per question -> full sample of 285 would take about 286 minutes
67.2s per question -> full sample of 285 would take about 319 minutes


In [ ]:
run(strat_sc, "sc", limit=5)

In [9]:
for sp, i, m in items[:3]:
    r, img = get_item(sp, i)
    print(S(r["question"]), "\n", strat_cot(S(r["question"]), img)["raw_response"], "\n---")

According to the image, in 2019, did Asian families have the highest real median household income in the United States, with White, Hispanic, and Black households following respectively? 
 The graph shows that in 2019, Asian families had the highest real median household income, followed by White, Hispanic, and Black households. The lines for each race clearly indicate this order.

Final answer: Yes 
---
According to the image, in 2019, did Black families have the highest real median household income in the United States, with White, Hispanic, and Asian households following respectively? 
 The graph shows that in 2019, Asian families had the highest real median household income, followed by White, not Hispanic families, then Hispanic (any race) families, and Black families last. 

Final answer: No 
---
According to the image, in 2019, did Black and White households represent the only racial groups with median incomes surpassing the national average in the United States, while Hispanic 

## 6. Full runs on the sample (slow; each is resume-safe)

### 6a. Chain-of-thought

In [ ]:
run(strat_cot, "cot")

### 6b. Chain-of-Verification

In [ ]:
run(strat_cove, "cove")

### 6c. Self-consistency (the slowest: K reasoning samples per question)

In [ ]:
run(strat_sc, "sc")

In [ ]:
for name, fn in [("cot", strat_cot), ("sc", strat_sc), ("cove", strat_cove)]:
    try:
        run(fn, name)
    except Exception as e:
        print(f"\n{name} stopped: {type(e).__name__}: {e}")
print("ALL DONE")


saved 285 rows to qwen25vl_cot.json


## 7. Metrics, confidence intervals and flip analysis

Metrics follow the official grouping used in Phase 2 (qAcc by category, subcategory, set, question; fAcc by category, subcategory, set, figure,
skipping VS figure 0). Confidence intervals come from resampling whole sets. First a check that this code reproduces your Phase 2 numbers.

In [ ]:
def is_ok(r):
    return S(r["model_prediction"]) == S(r["gt_answer"])

def metrics(rows):
    q, f = collections.defaultdict(list), collections.defaultdict(list)
    for r in rows:
        q[(r["category"], r["subcategory"], r["set_id"], r["question_id"])].append(is_ok(r))
        if not (r["category"] == "VS" and r["figure_id"] == "0"):
            f[(r["category"], r["subcategory"], r["set_id"], r["figure_id"])].append(is_ok(r))
    pct = lambda xs: 100 * sum(xs) / len(xs) if xs else float("nan")
    return dict(aAcc=pct([is_ok(r) for r in rows]), qAcc=pct([all(v) for v in q.values()]), fAcc=pct([all(v) for v in f.values()]),
                yes_rate=pct([S(r["model_prediction"]) == "1" for r in rows]), n=len(rows), n_pairs=len(q), n_figures=len(f))

def boot_ci(rows, B=300, seed=0):
    rg = random.Random(seed)
    bysets = collections.defaultdict(list)
    for r in rows:
        bysets[(r["category"], r["subcategory"], r["set_id"])].append(r)
    ks, res = list(bysets), []
    for _ in range(B):
        samp = []
        for j, k in enumerate(rg.choice(ks) for _ in ks):
            samp += [{**r, "set_id": r["set_id"] + f"#{j}"} for r in bysets[k]]
        res.append(metrics(samp))
    ci = lambda key: tuple(sorted(x[key] for x in res)[int(B * p)] for p in (0.025, 0.975))
    return {k: ci(k) for k in ("aAcc", "qAcc", "fAcc")}

# ---- self-check: reproduce Phase 2 numbers from the full zero-shot file ----
zs_all = json.load(open(ZS_FILE))
m_all = metrics(zs_all)
print("recomputed from Phase 2 file:", {k: round(v, 2) for k, v in m_all.items()})
print("expected (Phase 2): Qwen2.5-VL 69.80 / 37.58 / 43.06 | InternVL3 63.95 / 30.33 / 36.42 (aAcc / qAcc / fAcc), n_pairs 455, n_figures 346")

In [ ]:
zs = {rowkey(r["split"], r): r for r in zs_all}
zs_rows = [zs[k] for k in sorted(sample_keys)]

results = {"zero-shot": zs_rows}
for name in ("cot", "cove", "sc"):
    try:
        results[name] = [r for r in json.load(open(f"{MODEL_TAG}_{name}.json")) if r["key"] in sample_keys]
    except FileNotFoundError:
        print("missing:", name)

print(f"\nSample: {len(zs_rows)} questions\n")
print(f"{'strategy':<10}{'n':>5}  {'aAcc':>20}  {'qAcc':>20}  {'fAcc':>20}  {'yes%':>6}")
for name, rows in results.items():
    m, ci = metrics(rows), boot_ci(rows)
    cell = lambda k: f"{m[k]:5.1f} [{ci[k][0]:4.1f},{ci[k][1]:4.1f}]"
    print(f"{name:<10}{m['n']:>5}  {cell('aAcc'):>20}  {cell('qAcc'):>20}  {cell('fAcc'):>20}  {m['yes_rate']:6.1f}")

# unparseable and fallback rates
for name, rows in results.items():
    if name != "zero-shot":
        bad = sum(S(r["model_prediction"]) == "-1" for r in rows)
        fb = sum(r.get("parse_mode") == "fallback" for r in rows)
        print(f"{name}: unparseable {bad}/{len(rows)}, fallback-parsed {fb}")

In [ ]:
# Flip analysis against zero-shot: which questions change, and in which direction
zs_ok = {r["key"] if "key" in r else rowkey(r["split"], r): is_ok(r) for r in zs_rows}
print(f"{'strategy':<8} {'wrong->right':>13} {'right->wrong':>13} {'net':>5}")
for name, rows in results.items():
    if name == "zero-shot":
        continue
    w2r = sum(1 for r in rows if is_ok(r) and not zs_ok[r["key"]])
    r2w = sum(1 for r in rows if not is_ok(r) and zs_ok[r["key"]])
    print(f"{name:<8} {w2r:>13} {r2w:>13} {w2r - r2w:>+5}")

# Per-subcategory view (aAcc / qAcc / fAcc)
subs = sorted({(r["category"], r["subcategory"]) for r in zs_rows})
print(f"\n{'subcategory':<12}" + "".join(f"{n:>22}" for n in results))
for c, s in subs:
    line = f"{c}_{s:<10}"
    for name, rows in results.items():
        mm = metrics([r for r in rows if (r["category"], r["subcategory"]) == (c, s)])
        line += f"{mm['aAcc']:7.1f}/{mm['qAcc']:5.1f}/{mm['fAcc']:5.1f}   "
    print(line)

In [ ]:
# Download the results to attach to the report (or upload them back to Claude)
from google.colab import files
for name in ("cot", "cove", "sc"):
    try: files.download(f"{MODEL_TAG}_{name}.json")
    except Exception as e: print(name, "not downloaded:", e)

In [ ]:
from math import comb
gt_yes = lambda r: S(r["gt_answer"]) == "1"
pct = lambda xs: 100 * sum(xs) / len(xs)
print(f"{'strategy':<10}{'acc|GT=Yes':>12}{'acc|GT=No':>11}{'pred Yes%':>11}{'GT Yes%':>9}")
for name, rows in results.items():
    ys = [is_ok(r) for r in rows if gt_yes(r)]
    ns = [is_ok(r) for r in rows if not gt_yes(r)]
    print(f"{name:<10}{pct(ys):12.1f}{pct(ns):11.1f}{pct([S(r['model_prediction'])=='1' for r in rows]):11.1f}{pct([gt_yes(r) for r in rows]):9.1f}")

def sign_p(a, b):                      # exact two-sided sign test on flipped questions
    n, k = a + b, min(a, b)
    return min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n)
for name, rows in results.items():
    if name == "zero-shot": continue
    w2r = sum(1 for r in rows if is_ok(r) and not zs_ok[r["key"]])
    r2w = sum(1 for r in rows if not is_ok(r) and zs_ok[r["key"]])
    print(f"{name}: wrong->right {w2r}, right->wrong {r2w}, sign-test p = {sign_p(w2r, r2w):.3f}")